# 🧠 02 — Sarathi AI: Build FAISS Vectorstore

Loads all downloaded data, chunks it, creates embeddings, and saves FAISS index.

In [ ]:
!pip install -q langchain langchain-community faiss-cpu sentence-transformers pypdf

In [ ]:
import os, json
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

RAW = Path('data/raw')
os.makedirs('data/faiss_index', exist_ok=True)
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
all_docs = []

In [ ]:
# Load PDFs
for pdf in ['ipc.pdf','tourism_stats.pdf']:
    p = RAW / pdf
    if p.exists():
        docs = PyPDFLoader(str(p)).load()
        chunks = splitter.split_documents(docs)
        all_docs += chunks
        print(f'{pdf}: {len(chunks)} chunks')

In [ ]:
# Load text files
for txt in ['gita.txt','quran_english.txt','ramayana.txt','wiki_india.txt']:
    p = RAW / txt
    if p.exists():
        docs = TextLoader(str(p), encoding='utf-8').load()
        chunks = splitter.split_documents(docs)
        all_docs += chunks
        print(f'{txt}: {len(chunks)} chunks')

In [ ]:
# Load JSON structured data (hotels, places, routes)
def json_to_docs(path, label):
    with open(path) as f: items = json.load(f)
    docs = []
    for item in items:
        text = label + ':\n' + '\n'.join(f'{k}: {v}' for k,v in item.items() if v)
        docs.append(Document(page_content=text, metadata={'source': path, 'type': label}))
    return docs

for fname, label in [('kolkata_hotels.json','Hotel'),('tourist_places.json','Tourist Place'),('kolkata_routes.json','Travel Route')]:
    p = RAW / fname
    if p.exists():
        d = json_to_docs(str(p), label)
        all_docs += d
        print(f'{fname}: {len(d)} docs')

In [ ]:
print(f'Total chunks: {len(all_docs)}')
print('Building FAISS index...')
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', encode_kwargs={'normalize_embeddings':True})
db = FAISS.from_documents(all_docs, embeddings)
db.save_local('data/faiss_index')
print('✅ FAISS saved!')

In [ ]:
# Test queries
queries = ['Best budget hotel Kolkata under 2000','IPC Section 302','Howrah Bridge timing','Bhagavad Gita karma']
for q in queries:
    r = db.similarity_search(q, k=1)
    print(f'Q: {q}')
    print(f'A: {r[0].page_content[:300]}\n')